# BantAI — Fine-tuning XLM-RoBERTa for SMS Smishing Detection

**Sprint 2 · Track B (AI/ML) · WBS 2.3.4**

This notebook trains the smishing classifier that sorts Philippine SMS messages
into **Ham** (legitimate), **Spam** (unsolicited but honest marketing) and
**Scam** (fraud). The trained model is saved to `models/xlm-roberta-smishing/`,
which the FastAPI inference service loads automatically.

### Before you start
1. **Runtime → Change runtime type → T4 GPU** (the free tier is enough).
2. Have `bantai_colab_package.zip` ready to upload (Import the Dataset, below).

Expected total runtime on a T4: **~20–30 minutes.**

> **Re-running after a previous attempt?** Just run the cells again — the upload
> step deletes any package left over from the earlier run first, so a re-run
> cannot silently train on a stale copy of the data.

> **Privacy note.** The dataset contains real SMS messages collected from
> participants. Every preview in this notebook prints **masked** text only
> (`<URL>`, `<PHONE>`, `<AMOUNT>`, `<OTP>`), which is also exactly what the
> model itself sees during training.

## How This Notebook Follows Our Class Workflow

The steps are the same ones we use in class — **Import → Explore → Define model
→ Train → Evaluate model → Confusion matrix** — applied to text instead of
numeric columns. The table below maps each class concept to what this notebook
does, since the model is a language model rather than a Keras `Sequential`:

| In class | In this notebook | Why it differs |
|---|---|---|
| `Sequential()` with Dense layers | XLM-RoBERTa + a classification head | The input is Tagalog/English text, not numeric features, so a pre-trained language model is used instead of layers built from scratch |
| One neuron per class, **softmax** output | Three outputs (Ham, Spam, Scam), **softmax** | Same idea: multi-class classification, one output per class |
| `sparse_categorical_crossentropy` | Cross-entropy, **weighted by class** | Same loss family; the weights make mistakes on the rare Scam class cost more, because the dataset is imbalanced |
| `adam` optimizer | **AdamW** | Adam with weight decay, the standard choice for fine-tuning this model |
| `validation_split` | Stratified 80/20 split, shuffled with a fixed seed | "Randomize data", with the class balance preserved in both halves |
| Epochs, batch size | 4 epochs, batch size 16 | Same meaning |
| "Training accuracy doesn't matter, testing accuracy does" | Reported on held-out data only | The same rule; a separate frozen test set is also kept for the final thesis number |

# Check the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU attached. Go to Runtime > Change runtime type > T4 GPU, '
        'then re-run this cell. (Training on CPU would take many hours.)'
    )

print('GPU :', torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# Import Python Libraries

Colab already ships torch, pandas, numpy, scikit-learn and matplotlib, so only
the HuggingFace stack is installed here. The versions are pinned exactly: an
unpinned install once pulled a release that had removed an argument the training
code uses, and the run crashed halfway through.

In [ ]:
%pip install -q 'transformers==5.17.0' 'datasets==5.0.1' 'accelerate==1.15.0' 'huggingface-hub==1.31.0' sentencepiece

import matplotlib.pyplot as plt
import pandas as pd
import transformers, datasets

print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)
print('pandas      ', pd.__version__)

# Import the Dataset

Run the cell, then pick the package from your computer (it lives in `ai/colab/`
in the repo): **`bantai_colab_package_A.zip`** (with the reviewed synthetic scams)
or **`bantai_colab_package_B.zip`** (real messages only). The variant is stamped
inside the zip, so the trained model is saved under its own name either way.

Colab does **not** overwrite an existing file — a second upload of the same name
arrives as `bantai_colab_package (1).zip`. Any earlier copy is therefore deleted
first, and the filename the picker actually returns is what gets unpacked.

In [ ]:
import glob
import os

from google.colab import files

for stale in glob.glob('/content/bantai_colab_package*.zip'):
    os.remove(stale)
    print('removed stale upload:', stale)

uploaded = files.upload()
PACKAGE = '/content/' + list(uploaded)[0]
print('\nwill unpack:', PACKAGE)

# Unpack the Dataset Package

In [ ]:
import os
import shutil
import sys
import zipfile

shutil.rmtree('/content/bantai_ai', ignore_errors=True)
with zipfile.ZipFile(PACKAGE) as z:
    z.extractall('/content/bantai_ai')
%cd /content/bantai_ai

sys.path.insert(0, '.')
VARIANT = open('VARIANT.txt').read().split()[0] if os.path.exists('VARIANT.txt') else ''
print('unpacked and ready', f'-- variant {VARIANT}' if VARIANT else '')

# Explore Data

In [ ]:
from preprocessing import preprocess

dfsms = pd.read_csv('datasets/labeled/bantai_labeled.csv')
dfsms.shape

In [ ]:
dfsms.count()

In [ ]:
# Masked preview only -- the raw message text is never displayed.
preview = dfsms[['text', 'label']].head(5).copy()
preview['text'] = preview['text'].apply(preprocess)
preview

**How many messages are there in each class?**

In [ ]:
class_counts = dfsms['label'].value_counts()
print(class_counts)
print()
print((100 * class_counts / len(dfsms)).round(2), '%', sep='')

class_counts.plot(kind='bar', color=['#4c72b0', '#dd8452', '#c44e52'])
plt.title('Number of Messages per Class')
plt.xlabel('Class')
plt.ylabel('Number of Messages')
plt.xticks(rotation=0)
plt.show()

**Explanation:** Based on the conducted data analysis, it can be concluded that
the dataset is **imbalanced**: Ham is the majority class, while Scam is the
smallest. This is the reason the model is evaluated on **macro-F1** rather than
accuracy — a model that simply guessed "Ham" for every message would still score
a high accuracy while being useless at the task the system exists for. The
imbalance is also handled during training through class-weighted loss, so that a
mistake on the rare Scam class costs the model more than a mistake on Ham.

**How long are the messages in each class?**

In [ ]:
dfsms['word_count'] = dfsms['text'].str.split().str.len()
print(dfsms.groupby('label')['word_count'].describe()[['mean', '50%', 'max']].round(2))

dfsms.boxplot(column='word_count', by='label', grid=False)
plt.title('Message Length by Class')
plt.suptitle('')
plt.xlabel('Class')
plt.ylabel('Number of Words')
plt.show()

**Explanation:** Based on the conducted data analysis, it can be concluded that
message length alone does not separate the three classes — the distributions
overlap heavily. This supports the use of a language model that reads the actual
wording, rather than a simpler rule based on message length or keyword counts.

# Check for Null Values

In [ ]:
dfsms[['text', 'label']].isnull().sum()

**Explanation:** Based on the conducted data analysis, it can be concluded that
there are no missing messages or missing labels in the dataset, so no rows have
to be dropped or imputed before training.

# Handle Duplicates

Duplicates are removed on the **masked** form of the message, not the raw text,
because masking is what the model actually sees. Two different raw messages such
as `...claim at bit.ly/abc7` and `...claim at bit.ly/abc8` both become
`...claim at <URL>` once masked. If they were treated as different rows, the same
message could land in both the training and the validation set, and the model
would be graded on text it had already memorised.

In [ ]:
masked = dfsms['text'].apply(preprocess)
print('rows before de-duplication:', len(masked))
print('unique masked messages   :', masked.nunique())
print('duplicates removed       :', len(masked) - masked.nunique())

**Explanation:** Based on the conducted data analysis, it can be concluded that a
meaningful number of rows collapse onto an identical masked message. Removing
them is what keeps the validation score honest; an earlier version of this
pipeline that de-duplicated on raw text instead was found to be scoring the model
on messages it had already seen during training.

# Preprocess the Text (Privacy Masking and NFKC Normalization)

The same `preprocess()` function runs at training time and at inference time, so
the text the model is trained on always matches the text it is asked to classify
later. It normalises Unicode (NFKC) and replaces personal or variable details
with placeholders.

In [ ]:
# Synthetic examples -- not real participant messages.
examples = [
    'Claim your P5,000 reward now at http://promo-gcash.xyz code 483920',
    'Hi po, magkano po ang delivery fee? Salamat!',
    'Your account will be deactivated. Verify at bit.ly/verify2026 or call 09171234567',
]

for text in examples:
    print('raw   :', text)
    print('masked:', preprocess(text))
    print()

**Explanation:** Based on the conducted data analysis, it can be concluded that
masking removes the parts of a message that identify a person or that change from
one copy of a scam to the next — links, phone numbers, amounts and one-time codes
— while keeping the wording that actually signals fraud. This protects the
privacy of the participants whose messages were collected, and it prevents the
model from memorising a specific link instead of learning the pattern.

# Randomize and Split the Data

The rows are **shuffled** before splitting, so the order they were collected in
cannot leak into the result — the dataset was assembled source by source, so an
unshuffled split would put whole sources on one side.

The split is also **stratified**: each class keeps the same proportion in the
training and validation halves. And it uses a **fixed seed (42)**, so the shuffle
is random but repeatable — re-running gives the same split, which is what makes
two training runs comparable.

In [ ]:
from training.config import ID2LABEL, TrainingConfig
from training.dataset import load_split

train_texts, val_texts, train_labels, val_labels = load_split(TrainingConfig())

print('train rows:', len(train_texts))
print('val rows  :', len(val_texts))
print()

# Stratification check: the class balance should match in both halves.
for name, labels in (('train', train_labels), ('val', val_labels)):
    counts = pd.Series([ID2LABEL[i] for i in labels]).value_counts(normalize=True).round(3)
    print(name, dict(counts))

**Explanation:** Based on the conducted data analysis, it can be concluded that
the dataset is now ready for training: roughly 80% of the unique masked messages
are used to fit the model, and the remaining 20% are held back to measure it.
Note that a separate, permanently frozen holdout set is kept outside this file
entirely and is used for the final evaluation reported in the thesis.

# Define the Model and Train It

The model is **XLM-RoBERTa-base**, a multilingual language model pre-trained on
100 languages including Filipino, which is why it handles Tagalog, English and
Taglish messages. A 3-class classification head is fine-tuned on top of it.

| Setting | Value |
|---|---|
| Optimizer | AdamW, learning rate 2e-5 |
| Epochs | 4 |
| Batch size | 16 |
| Maximum length | 128 tokens |
| Loss | Class-weighted cross-entropy |
| Model selection | Best epoch by macro-F1 |

Training is run through the project's own `training/train.py` rather than being
rewritten here, so that the notebook and the production pipeline can never drift
apart and produce different models.

In [ ]:
!python -m training.train

**Explanation:** Based on the conducted training run, it can be concluded that
the model improves across epochs and that the **best epoch by macro-F1** is the
one saved — not simply the last one. Watching macro-F1 rather than accuracy
matters here for the reason shown in the class distribution above: accuracy is
flattered by the majority Ham class, while macro-F1 weighs all three classes
equally.

# Evaluate the Model

The saved model is scored on the same validation split (fixed seed 42). The
`support` column must sum to the number of validation rows printed earlier — if
it does not, an outdated package was uploaded.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

cfg = TrainingConfig()
tok = AutoTokenizer.from_pretrained(cfg.output_dir)
model = AutoModelForSequenceClassification.from_pretrained(cfg.output_dir)
model = model.cuda().eval()

preds = []
with torch.no_grad():
    for i in range(0, len(val_texts), 64):
        batch = tok(
            val_texts[i:i + 64],
            truncation=True,
            max_length=cfg.max_length,
            padding=True,
            return_tensors='pt',
        ).to('cuda')
        preds.extend(model(**batch).logits.argmax(-1).cpu().tolist())

names = [ID2LABEL[i] for i in sorted(ID2LABEL)]
print(classification_report(val_labels, preds, target_names=names, digits=4))

**Explanation:** Based on the conducted evaluation, it can be concluded that the
model performs strongest on Ham and weakest on Scam, which follows the amount of
training data available for each class. The most important figure for this system
is **Scam recall** — the share of real fraudulent messages the model catches —
because a missed scam can cost a user money, while a misfiled promotional message
only causes mild annoyance.

# Confusion Matrix

In [ ]:
cm = confusion_matrix(val_labels, preds)
print(pd.DataFrame(cm, index=names, columns=names))

fig, ax = plt.subplots()
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(names)), names)
ax.set_yticks(range(len(names)), names)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
ax.set_title('Confusion Matrix')
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, cm[i][j], ha='center', va='center',
                color='white' if cm[i][j] > cm.max() / 2 else 'black')
fig.colorbar(im)
plt.show()

**Explanation:** Based on the conducted data analysis, it can be concluded that
the errors are not spread evenly. The cell that matters most is **Scam predicted
as Ham** — a fraudulent message delivered to the user as if it were safe. The
opposite error, **Ham predicted as Scam**, is a legitimate message wrongly
flagged, which harms the usefulness of the app rather than the safety of the
user. Both are tracked separately in the thesis for this reason.

# Save the Trained Model

The model is about 1.1 GB. Saving to Google Drive is the reliable route; the
direct browser download often stalls at this size.

**Colab deletes everything when the session ends — do not skip this step.**

In [ ]:
import datetime

# Dated name, so a new run can never overwrite an older model on Drive.
# (Every earlier run saved as bantai_model.zip, and the 08-17/08-26 runs were lost that way.)
RUN = datetime.date.today().isoformat() + '-colab' + (f'-{VARIANT}' if VARIANT else '')
ZIP = f'/content/bantai_model_{RUN}.zip'

# checkpoint-* folders only matter for resuming a run; leaving them out
# makes the download ~1 GB instead of ~2.3 GB.
!zip -qr {ZIP} models/xlm-roberta-smishing -x '*/checkpoint-*'
!ls -lh {ZIP}

from google.colab import drive

drive.mount('/content/drive')
DEST = f'/content/drive/MyDrive/bantai/models/{RUN}'
!mkdir -p '{DEST}'
!cp {ZIP} '{DEST}/'
print(f'Saved to Google Drive: MyDrive/bantai/models/{RUN}/bantai_model_{RUN}.zip')


Optional direct download, instead of or in addition to Drive:

In [ ]:
from google.colab import files

files.download(ZIP)

**CONCLUSION:**

Based on the conducted training and evaluation, it can be concluded that
XLM-RoBERTa can be fine-tuned to classify Philippine SMS messages into Ham, Spam
and Scam with strong overall performance, while Scam, the class with the fewest real
examples, has the lowest F1. The class
imbalance was addressed through class-weighted loss, and the evaluation was kept
honest by de-duplicating messages on their masked form so that the model is never
graded on text it has already seen. Privacy was preserved throughout by masking
links, phone numbers, amounts and one-time codes before the model reads any
message. Future improvement of this model therefore depends mainly on collecting
more genuine examples of the rarer scam types, rather than on further tuning of
the training settings.

# Back on Your Laptop

**Do not unzip over `ai/models/xlm-roberta-smishing/`.** That folder is the model
currently serving users. A new model only replaces it after it passes the holdout
test and the promotion gate (Scam-recall floor included).

Unzip the downloaded file into a *new* candidate folder instead, named by the date
you trained it:

```
ai/models/retraining_runs/<YYYY-MM-DD>-colab-<variant>/candidate/
```

The zip contains `models/xlm-roberta-smishing/...`, so move the *contents* of that
inner folder into `candidate/`. Then, from `ai/`:

```
.venv/Scripts/python.exe scripts/evaluate_holdout.py --model-dir models/retraining_runs/<YYYY-MM-DD>-colab/candidate
```

Promote only if it passes. `ai/models/` is git-ignored, so the weights stay out of
the repository.
